In [1]:
import sys
import os
from pathlib import Path

# Walk upward until we find the folder that contains "src"
p = Path.cwd().resolve()
while p != p.parent and not (p / "src").exists():
    p = p.parent

if not (p / "src").exists():
    raise RuntimeError("Could not find repo root (folder containing 'src').")

if str(p) not in sys.path:
    sys.path.insert(0, str(p))

print("Repo root set to:", p)


Repo root set to: /home/iauger/projects/dsci632-project


In [2]:
import os, subprocess
import sys, platform

print("python:", sys.executable)
print("platform:", platform.platform())
print("cwd:", os.getcwd())
print("home:", str(Path.home()))

assert "/home/" in str(Path.home()), "Not running in WSL home"
assert "mnt/c" not in os.getcwd().lower(), "CWD is on Windows mount; use ~/projects/... instead"
print("JAVA_HOME:", os.environ.get("JAVA_HOME"))
print("which java:", subprocess.check_output(["which","java"]).decode().strip())

python: /home/iauger/projects/dsci632-project/.venv/bin/python
platform: Linux-5.15.167.4-microsoft-standard-WSL2-x86_64-with-glibc2.39
cwd: /home/iauger/projects/dsci632-project/notebooks
home: /home/iauger
JAVA_HOME: /usr/lib/jvm/java-17-openjdk-amd64
which java: /usr/lib/jvm/java-17-openjdk-amd64/bin/java


In [3]:
from src.config import load_settings 

s = load_settings()
print("ENV:", s.env)
print("RAW recipes:", s.raw_recipes_path)
print("RAW interactions:", s.raw_interactions_path)
print("PROCESSED:", s.processed_dir)
print("BRONZE:", s.bronze_dir)
print("SILVER:", s.silver_dir)
print("GOLD:", s.gold_dir)

ENV: local
RAW recipes: /home/iauger/projects/dsci632-project/data/raw/RAW_recipes.csv
RAW interactions: /home/iauger/projects/dsci632-project/data/raw/RAW_interactions.csv
PROCESSED: /home/iauger/projects/dsci632-project/data/processed
BRONZE: /home/iauger/projects/dsci632-project/data/processed/bronze
SILVER: /home/iauger/projects/dsci632-project/data/processed/silver
GOLD: /home/iauger/projects/dsci632-project/data/processed/gold


In [4]:
from src.spark.session import get_spark
    
spark = get_spark(app_name="testing-spark-session", debug=True)

print("spark.version:", spark.version)
print("spark.driver.host:", spark.conf.get("spark.driver.host", "NOT SET"))
print("spark.local.ip:", spark.conf.get("spark.local.ip", "NOT SET"))

your 131072x1 screen size is bogus. expect trouble
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/26 00:41:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


spark.version: 3.5.1
spark.driver.host: 127.0.0.1
spark.local.ip: 127.0.0.1


In [5]:
import shutil
from pathlib import Path

targets = [
    Path(s.bronze_recipes_path),
    Path(s.bronze_interactions_path),
    Path(s.silver_recipes_path),
    Path(s.silver_interactions_path),
    Path(s.gold_ve_path),
    Path(s.gold_cf_path),
]

for t in targets:
    if t.exists():
        shutil.rmtree(t)
        print("deleted:", t)
    else:
        print("missing (ok):", t)

deleted: /home/iauger/projects/dsci632-project/data/processed/bronze/recipes_raw.parquet
deleted: /home/iauger/projects/dsci632-project/data/processed/bronze/interactions_raw.parquet
deleted: /home/iauger/projects/dsci632-project/data/processed/silver/recipes_clean.parquet
deleted: /home/iauger/projects/dsci632-project/data/processed/silver/interactions_clean.parquet
deleted: /home/iauger/projects/dsci632-project/data/processed/gold/modeling_ve.parquet
missing (ok): /home/iauger/projects/dsci632-project/data/processed/gold/modeling_cf.parquet


In [6]:
recipes_raw = (
    spark.read
    .option("header", "true")
    .option("multiLine", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("mode", "PERMISSIVE")
    .option("enforceSchema", "false")
    .csv(s.raw_recipes_path)
)

interactions_raw = (
    spark.read
    .option("header", "true")
    .option("multiLine", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("mode", "PERMISSIVE")
    .option("enforceSchema", "false")
    .csv(s.raw_interactions_path)
)

recipes_raw.write.mode("overwrite").parquet(s.bronze_recipes_path)
interactions_raw.write.mode("overwrite").parquet(s.bronze_interactions_path)

In [7]:
recipes_b = spark.read.parquet(s.bronze_recipes_path)

# id should be numeric-ish and definitely not long sentences
recipes_b.select("id", "nutrition").show(10, truncate=False)

# sanity: find “bad id” rows (contains spaces or punctuation typical of text)
from pyspark.sql import functions as F
bad = recipes_b.filter(F.col("id").rlike(r"[A-Za-z]|\s"))
bad.select("id", "nutrition").show(20, truncate=False)
print("bad id count:", bad.count())

+------+---------------------------------------------------+
|id    |nutrition                                          |
+------+---------------------------------------------------+
|137739|[51.5, 0.0, 13.0, 0.0, 2.0, 0.0, 4.0]              |
|31490 |[173.4, 18.0, 0.0, 17.0, 22.0, 35.0, 1.0]          |
|112140|[269.8, 22.0, 32.0, 48.0, 39.0, 27.0, 5.0]         |
|59389 |[368.1, 17.0, 10.0, 2.0, 14.0, 8.0, 20.0]          |
|44061 |[352.9, 1.0, 337.0, 23.0, 3.0, 0.0, 28.0]          |
|5289  |[160.2, 10.0, 55.0, 3.0, 9.0, 20.0, 7.0]           |
|25274 |[380.7, 53.0, 7.0, 24.0, 6.0, 24.0, 6.0]           |
|67888 |[1109.5, 83.0, 378.0, 275.0, 96.0, 86.0, 36.0]     |
|70971 |[4270.8, 254.0, 1306.0, 111.0, 127.0, 431.0, 220.0]|
|75452 |[2669.3, 160.0, 976.0, 107.0, 62.0, 310.0, 138.0]  |
+------+---------------------------------------------------+
only showing top 10 rows

+---+---------+
|id |nutrition|
+---+---------+
+---+---------+

bad id count: 0


In [8]:
from src.spark.ingestion.spark.transforms.recipes_spark import clean_recipes     
from src.spark.ingestion.spark.transforms.interactions_spark import clean_interactions 

recipes_b = spark.read.parquet(s.bronze_recipes_path)
interactions_b = spark.read.parquet(s.bronze_interactions_path)

recipes_s = clean_recipes(recipes_b)
interactions_s = clean_interactions(interactions_b)

recipes_s.write.mode("overwrite").parquet(s.silver_recipes_path)
interactions_s.write.mode("overwrite").parquet(s.silver_interactions_path)

print("silver written:")
print(" -", s.silver_recipes_path)
print(" -", s.silver_interactions_path)

silver written:
 - /home/iauger/projects/dsci632-project/data/processed/silver/recipes_clean.parquet
 - /home/iauger/projects/dsci632-project/data/processed/silver/interactions_clean.parquet


In [9]:
interactions_s = spark.read.parquet(s.silver_interactions_path)
interactions_s.printSchema()

root
 |-- user_id: string (nullable = true)
 |-- recipe_id: string (nullable = true)
 |-- date: date (nullable = true)
 |-- rating: double (nullable = true)
 |-- review: string (nullable = true)
 |-- review_clean: string (nullable = true)



In [10]:
interactions_s.show(10, truncate=False)

+----------+---------+----------+------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|user_id  

In [11]:
interactions_s.printSchema()
interactions_s.groupBy("rating").count().orderBy("rating").show()

root
 |-- user_id: string (nullable = true)
 |-- recipe_id: string (nullable = true)
 |-- date: date (nullable = true)
 |-- rating: double (nullable = true)
 |-- review: string (nullable = true)
 |-- review_clean: string (nullable = true)

+------+------+
|rating| count|
+------+------+
|   1.0| 12818|
|   2.0| 14123|
|   3.0| 40855|
|   4.0|187360|
|   5.0|816364|
+------+------+



In [12]:
recipes_s.printSchema()

root
 |-- name: string (nullable = true)
 |-- id: string (nullable = true)
 |-- minutes: double (nullable = true)
 |-- contributor_id: string (nullable = true)
 |-- submitted: timestamp (nullable = true)
 |-- tags: string (nullable = true)
 |-- nutrition: string (nullable = true)
 |-- n_steps: double (nullable = true)
 |-- steps: string (nullable = true)
 |-- description: string (nullable = true)
 |-- ingredients: string (nullable = true)
 |-- n_ingredients: double (nullable = true)
 |-- ingredients_clean: string (nullable = false)
 |-- steps_clean: string (nullable = false)
 |-- tags_clean: string (nullable = false)
 |-- calories: double (nullable = true)
 |-- fat: double (nullable = true)
 |-- sugar: double (nullable = true)
 |-- sodium: double (nullable = true)
 |-- protein: double (nullable = true)
 |-- saturated_fat: double (nullable = true)
 |-- carbs: double (nullable = true)
 |-- description_clean: string (nullable = false)



In [13]:
from pyspark.sql import functions as F

recipes_s = spark.read.parquet(s.silver_recipes_path)
interactions_s = spark.read.parquet(s.silver_interactions_path)

print("recipes_s:", recipes_s.count(), "cols:", len(recipes_s.columns))
print("interactions_s:", interactions_s.count(), "cols:", len(interactions_s.columns))

# check null rates on cleaned fields
recipes_s.select(
    F.sum(F.col("ingredients_clean").isNull().cast("int")).alias("null_ingredients_clean"),
    F.sum(F.col("steps_clean").isNull().cast("int")).alias("null_steps_clean"),
    F.sum(F.col("tags_clean").isNull().cast("int")).alias("null_tags_clean"),
    F.sum(F.col("description_clean").isNull().cast("int")).alias("null_description_clean"),
).show()

interactions_s.select(
    F.sum(F.col("review_clean").isNull().cast("int")).alias("null_review_clean"),
    F.sum(F.col("rating").isNull().cast("int")).alias("null_rating"),
).show()

# confirm rating 0 removed
interactions_s.groupBy("rating").count().orderBy("rating").show(10)

recipes_s: 231637 cols: 23
interactions_s: 1071520 cols: 6
+----------------------+----------------+---------------+----------------------+
|null_ingredients_clean|null_steps_clean|null_tags_clean|null_description_clean|
+----------------------+----------------+---------------+----------------------+
|                     0|               0|              0|                     0|
+----------------------+----------------+---------------+----------------------+

+-----------------+-----------+
|null_review_clean|null_rating|
+-----------------+-----------+
|                0|          0|
+-----------------+-----------+

+------+------+
|rating| count|
+------+------+
|   1.0| 12818|
|   2.0| 14123|
|   3.0| 40855|
|   4.0|187360|
|   5.0|816364|
+------+------+



In [14]:
print(s.gold_reviews_path)

/home/iauger/projects/dsci632-project/data/processed/gold/modeling_reviews.parquet


In [15]:
from src.spark.ingestion.spark.transforms.merge_spark import build_gold_recipes, build_gold_reviews
from pyspark import StorageLevel

recipes_s = spark.read.parquet(s.silver_recipes_path)
interactions_s = spark.read.parquet(s.silver_interactions_path)

gold_reviews = build_gold_reviews(interactions_s)
gold_recipes = build_gold_recipes(recipes_s)

gold_reviews.write.mode("overwrite").parquet(s.gold_reviews_path)
gold_recipes.write.mode("overwrite").parquet(s.gold_recipe_path)

In [16]:
from src.spark.ingestion.spark.transforms.merge_spark import build_gold_ve

gold_ve = build_gold_ve(gold_recipes, gold_reviews)

gold_ve.coalesce(32).write.mode("overwrite").parquet(s.gold_ve_path)
print("Wrote gold_ve")

Wrote gold_ve


In [17]:
gold_reviews.schema

StructType([StructField('review_key', StringType(), True), StructField('user_id', StringType(), True), StructField('recipe_id', StringType(), True), StructField('date', DateType(), True), StructField('rating', DoubleType(), True), StructField('liked', IntegerType(), True), StructField('review_clean', StringType(), True)])

In [18]:
# Row counts and column count review
print("DF SHAPE")
print("VE:", gold_ve.count(), "cols:", len(gold_ve.columns))

# Like distribution
print(40 * "-")
print("\nLIKE DISTRIBUTION")
gold_ve.groupBy("liked").count().show()

tot = gold_ve.count()
pos = gold_ve.filter(F.col("liked") == 1).count()
pct_pos = pos / tot * 100
print(f"Positive class (liked=1) count: {pos} / {tot} ({pct_pos:.2f}%)")

# Rating distribution
print(40 * "-")
print("\nRATING DISTRIBUTION")
gold_ve.groupBy("rating").count().orderBy("rating").show()

# Top reviewed recipes
print(40 * "-")
print("\nTOP 10 RECIPES BY REVIEW COUNT")
top10 = (
    gold_ve.groupBy("recipe_id", "name")
         .agg(
             F.count("*").alias("n_reviews"),
             F.round(F.avg("rating"), 3).alias("avg_rating"),
             F.sum(F.col("liked").cast("int")).alias("n_liked"),
         )
         .orderBy(F.desc("n_reviews"), F.desc("avg_rating"))
         .limit(10)
)
top10.show(truncate=60)


# Review length distribution (chars)
print(40 * "-")
print("\nREVIEW LENGTH (chars) SUMMARY")
review_len = gold_ve.withColumn("review_len", F.length(F.coalesce(F.col("review_clean"), F.lit(""))))

review_len.select(
    F.min("review_len").alias("min_len"),
    F.round(F.avg("review_len"), 2).alias("avg_len"),
    F.max("review_len").alias("max_len"),
).show()

# Percentiles (more informative than min/max)
print(40 * "-")
print("\nREVIEW LENGTH (chars) PERCENTILES")
review_len.selectExpr("percentile_approx(review_len, array(0.5, 0.9, 0.95, 0.99)) as p").show(truncate=False)

# Null counts (VE required cols)
print(40 * "-")
print("\nNULL COUNTS (VE REQUIRED)")
required_ve = [
    "user_id","recipe_id","name","rating","liked","date","review_clean",
    "ingredients_clean","steps_clean","tags_clean","description_clean",
    "minutes","n_steps","n_ingredients",
    "calories","fat","sugar","sodium","protein","saturated_fat","carbs",
]
existing_required_ve = [c for c in required_ve if c in gold_ve.columns]

gold_ve.select(
    *[F.sum(F.col(c).isNull().cast("int")).alias(f"null_{c}") for c in existing_required_ve]
).show(truncate=False)

DF SHAPE
VE: 1071520 cols: 22
----------------------------------------

LIKE DISTRIBUTION
+-----+-------+
|liked|  count|
+-----+-------+
|    1|1003724|
|    0|  67796|
+-----+-------+

Positive class (liked=1) count: 1003724 / 1071520 (93.67%)
----------------------------------------

RATING DISTRIBUTION
+------+------+
|rating| count|
+------+------+
|   1.0| 12818|
|   2.0| 14123|
|   3.0| 40855|
|   4.0|187360|
|   5.0|816364|
+------+------+

----------------------------------------

TOP 10 RECIPES BY REVIEW COUNT
+---------+-------------------------------------------------+---------+----------+-------+
|recipe_id|                                             name|n_reviews|avg_rating|n_liked|
+---------+-------------------------------------------------+---------+----------+-------+
|    27208|                       to die for crock pot roast|     1496|      4.59|   1338|
|    89204|crock pot chicken with black beans   cream cheese|     1488|     4.478|   1306|
|     2886|        

In [23]:
from src.spark.labeling.zero_shot import ZeroShotConfig, prepare_zero_shot_input_from_gold
from src.spark.labeling.zero_shot import run_zero_shot_from_input_parquet
import pandas as pd
import time

cfg = ZeroShotConfig(
    taxonomy_version="v1",
    min_tokens=15,
    sample_n=7500,
    batch_size=16,
    sample_seed=42,
)

inp = prepare_zero_shot_input_from_gold(
    spark=spark,
    gold_reviews_path=s.gold_reviews_path,  
    cfg=cfg,
    processed_dir=s.processed_dir,
)

print("zero-shot input:", inp)

start = time.time()
out = run_zero_shot_from_input_parquet(inp, cfg, s.processed_dir)
elapsed = time.time() - start

print("Elapsed seconds:", elapsed)
print("Seconds per sample:", elapsed / cfg.sample_n)
print("zero-shot output:", out)

zero-shot input: /home/iauger/projects/dsci632-project/data/processed/labeling/zero_shot/zs_input.parquet


Loading weights:   0%|          | 0/231 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Zero-shot batches:   0%|          | 0/469 [00:00<?, ?batch/s]

Elapsed seconds: 34388.39166045189
Seconds per sample: 4.585118888060252
zero-shot output: /home/iauger/projects/dsci632-project/data/processed/labeling/zero_shot/zs_output.parquet


In [24]:
from src.spark.labeling.zero_shot import attach_zero_shot_labels_to_gold
labeled = attach_zero_shot_labels_to_gold(spark, s.gold_reviews_path, s.processed_dir, cfg)
print("labeled parquet:", labeled)

labeled parquet: /home/iauger/projects/dsci632-project/data/processed/labeling/zero_shot/labeled_gold_reviews.parquet


In [25]:
df_labeled = spark.read.parquet(labeled)
print("Labeled DF schema:")
df_labeled.printSchema()
print("Labeled DF sample:")
df_labeled.show(10, truncate=False)

Labeled DF schema:
root
 |-- review_key: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- recipe_id: string (nullable = true)
 |-- date: date (nullable = true)
 |-- rating: double (nullable = true)
 |-- liked: integer (nullable = true)
 |-- review_clean: string (nullable = true)
 |-- zs_labels: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- zs_scores_json: string (nullable = true)
 |-- zs_num_labels: long (nullable = true)
 |-- zs_max_score: double (nullable = true)
 |-- zs_used_fallback: boolean (nullable = true)
 |-- zs_fallback_choice: integer (nullable = true)
 |-- zs_fallback_choice_score: integer (nullable = true)
 |-- zs_fallback_candidate_max: double (nullable = true)

Labeled DF sample:
+----------------------------------------------------------------+----------+---------+----------+------+-----+--------------------------------------------------------------------------------------------------------------------------------------

In [26]:
import pandas as pd
import json
from collections import Counter, defaultdict

from src.spark.labeling.taxonomy import get_taxonomy

def run_zs_diagnostics_pandas(zs_out_path, taxonomy_version="v1", top_k_tags=20):
    pdf = pd.read_parquet(zs_out_path)
    tax = get_taxonomy(taxonomy_version)
    pol_map = {tid: t.polarity for tid, t in tax.items()}

    def normalize_labels(x):
        if x is None:
            return []
        if isinstance(x, float) and pd.isna(x):
            return []
        if isinstance(x, str):
            return [x] if x else []
        if hasattr(x, "tolist"):
            x = x.tolist()
        if isinstance(x, (list, tuple, set)):
            return [t for t in x if t is not None and not (isinstance(t, float) and pd.isna(t))]
        return []

    n = len(pdf)
    print("n_reviews:", n)
    print("avg_labels_per_review:", round(pdf["zs_num_labels"].mean(), 3))
    print("median_labels_per_review:", pdf["zs_num_labels"].median())
    print("fallback_rate:", round(pdf["zs_used_fallback"].mean(), 4))
    print("avg_max_score:", round(pdf["zs_max_score"].mean(), 4))

    # per-tag prevalence
    tag_ctr = Counter()
    for labels in pdf["zs_labels"]:
        tag_ctr.update(normalize_labels(labels))
    print(f"\nTop {top_k_tags} tags (% of reviews):")
    for tid, cnt in tag_ctr.most_common(top_k_tags):
        print(f"{tid:28s} {cnt/max(n, 1):.3f}  ({pol_map.get(tid,'?')})")

    # polarity distribution over ASSIGNED labels
    pol_ctr = Counter()
    for labels in pdf["zs_labels"]:
        for tid in normalize_labels(labels):
            pol_ctr[pol_map.get(tid, "unknown")] += 1
    total_assigned = sum(pol_ctr.values()) or 1
    print("\nPolarity distribution over assigned labels:")
    for pol in ["positive", "negative", "neutral", "unknown"]:
        print(f"{pol:10s} {pol_ctr[pol]/total_assigned:.3f}")

    # % reviews with >=1 tag of each polarity
    def has_pol(labels, pol):
        labels = normalize_labels(labels)
        return any(pol_map.get(t) == pol for t in labels)

    print("\nReview-level polarity presence:")
    print("pct_reviews_with_pos:", round(pdf["zs_labels"].apply(lambda x: has_pol(x, "positive")).mean(), 3))
    print("pct_reviews_with_neg:", round(pdf["zs_labels"].apply(lambda x: has_pol(x, "negative")).mean(), 3))
    print("pct_reviews_with_neu:", round(pdf["zs_labels"].apply(lambda x: has_pol(x, "neutral")).mean(), 3))

    # score means by polarity (all tag scores, not just assigned)
    pol_score_sum = defaultdict(float)
    pol_score_n = defaultdict(int)
    for s in pdf["zs_scores_json"]:
        if isinstance(s, dict):
            scores = s
        elif isinstance(s, str) and s.strip():
            scores = json.loads(s)
        else:
            scores = {}
        for tid, sc in scores.items():
            pol = pol_map.get(tid, "unknown")
            pol_score_sum[pol] += float(sc)
            pol_score_n[pol] += 1

    print("\nAverage raw score by polarity (all tag scores):")
    for pol in ["positive", "negative", "neutral", "unknown"]:
        if pol_score_n[pol]:
            print(f"{pol:10s} {pol_score_sum[pol]/pol_score_n[pol]:.4f}")

# Usage:
# run_zs_diagnostics_pandas("/home/iauger/.../zs_output.parquet", taxonomy_version="v1", top_k_tags=20)

run_zs_diagnostics_pandas(labeled, taxonomy_version=cfg.taxonomy_version, top_k_tags=20)

n_reviews: 7500
avg_labels_per_review: 3.237
median_labels_per_review: 3.0
fallback_rate: 0.0
avg_max_score: 0.9443

Top 20 tags (% of reviews):
delicious_tasty              0.781  (positive)
would_make_again             0.763  (positive)
family_hit                   0.672  (positive)
substitution_modification    0.441  (neutral)
easy_quick                   0.307  (positive)
ingredient_issue             0.071  (negative)
would_not_make_again         0.067  (negative)
bland_lacks_flavor           0.039  (negative)
moist_tender                 0.031  (positive)
time_consuming_complex       0.030  (negative)
crispy_crunchy               0.015  (positive)
too_sweet                    0.006  (negative)
dry                          0.005  (negative)
too_salty                    0.003  (negative)
too_spicy                    0.003  (negative)
mushy_soggy                  0.002  (negative)
too_acidic                   0.001  (negative)

Polarity distribution over assigned labels:
positive   0